In [2]:
import pandas as pd
import numpy as np

orders_data = [
    [101, "Laptop", 2, 120000, "Delivered"],
    [102, "Mobile", 1, 30000, "Pending"],
    [103, "Tablet", 3, 75000, "Delivered"],
    [104, "Monitor", 2, 40000, "Pending"],
    [105, "Keyboard", 5, 10000, "Delivered"],
    [106, "Headphones", 4, 20000, "Pending"],
    [107, "Laptop", 1, 60000, "Delivered"],
    [108, "Mobile", 2, 50000, "Pending"],
    [109, "Tablet", 1, 25000, "Delivered"],
    [110, "Monitor", 3, 60000, "Pending"]
]

columns = [
    "order_id",
    "product",
    "quantity",
    "order_amount",
    "delivery_status"
]

df = pd.DataFrame(orders_data, columns=columns)

df.to_csv("orders.csv", index=False)

print("CSV Dataset Created")
df.head()

CSV Dataset Created


,order_id,product,quantity,order_amount,delivery_status
0,101,Laptop,2,120000,Delivered
1,102,Mobile,1,30000,Pending
2,103,Tablet,3,75000,Delivered
3,104,Monitor,2,40000,Pending
4,105,Keyboard,5,10000,Delivered


In [15]:
from google.colab import files

files.download("orders.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
orders_df = pd.read_csv("orders.csv")

print("Dataset Loaded Successfully")
orders_df.head()

Dataset Loaded Successfully


,order_id,product,quantity,order_amount,delivery_status
0,101,Laptop,2,120000,Delivered
1,102,Mobile,1,30000,Pending
2,103,Tablet,3,75000,Delivered
3,104,Monitor,2,40000,Pending
4,105,Keyboard,5,10000,Delivered


In [4]:
cleaned_df = orders_df.dropna()

print("Data After Cleaning")
cleaned_df.head()

Data After Cleaning


,order_id,product,quantity,order_amount,delivery_status
0,101,Laptop,2,120000,Delivered
1,102,Mobile,1,30000,Pending
2,103,Tablet,3,75000,Delivered
3,104,Monitor,2,40000,Pending
4,105,Keyboard,5,10000,Delivered


In [5]:
filtered_df = cleaned_df[
    cleaned_df["delivery_status"] == "Delivered"
]

print("Filtered Data")
filtered_df

Filtered Data


,order_id,product,quantity,order_amount,delivery_status
0,101,Laptop,2,120000,Delivered
2,103,Tablet,3,75000,Delivered
4,105,Keyboard,5,10000,Delivered
6,107,Laptop,1,60000,Delivered
8,109,Tablet,1,25000,Delivered


In [6]:
filtered_df.to_csv(
    "cleaned_orders.csv",
    index=False
)

print("Cleaned CSV Saved")

Cleaned CSV Saved


In [7]:
from google.colab import files

files.download("cleaned_orders.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!pip install deltalake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.6 MB/s eta 0:00:00


In [9]:
from deltalake.writer import write_deltalake

write_deltalake(
    "orders_delta",
    filtered_df
)

print("Delta Table Created")

Delta Table Created


In [10]:
!zip -r orders_delta.zip orders_delta
files.download("orders_delta.zip")

  adding: orders_delta/ (stored 0%)
  adding: orders_delta/part-00000-898eeef9-71eb-4a76-b076-d5dfe6d605b0-c000.snappy.parquet (deflated 52%)
  adding: orders_delta/_delta_log/ (stored 0%)
  adding: orders_delta/_delta_log/00000000000000000000.json (deflated 60%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
!pip install pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OrdersAnalysis") \
    .getOrCreate()

In [12]:
spark_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("cleaned_orders.csv")

spark_df.show()

+--------+--------+--------+------------+---------------+
|order_id| product|quantity|order_amount|delivery_status|
+--------+--------+--------+------------+---------------+
|     101|  Laptop|       2|      120000|      Delivered|
|     103|  Tablet|       3|       75000|      Delivered|
|     105|Keyboard|       5|       10000|      Delivered|
|     107|  Laptop|       1|       60000|      Delivered|
|     109|  Tablet|       1|       25000|      Delivered|
+--------+--------+--------+------------+---------------+



In [13]:
from pyspark.sql.functions import *

print("Total Orders:", spark_df.count())
spark_df.select(avg("order_amount").alias("Average_Order_Amount")).show()
spark_df.orderBy("order_amount",ascending=False).show(5)
spark_df.groupBy("product").agg(sum("order_amount").alias("Total_Revenue")).orderBy("Total_Revenue",ascending=False).show()
spark_df.groupBy("delivery_status").count().show()

Total Orders: 5
+--------------------+
|Average_Order_Amount|
+--------------------+
|             58000.0|
+--------------------+

+--------+--------+--------+------------+---------------+
|order_id| product|quantity|order_amount|delivery_status|
+--------+--------+--------+------------+---------------+
|     101|  Laptop|       2|      120000|      Delivered|
|     103|  Tablet|       3|       75000|      Delivered|
|     107|  Laptop|       1|       60000|      Delivered|
|     109|  Tablet|       1|       25000|      Delivered|
|     105|Keyboard|       5|       10000|      Delivered|
+--------+--------+--------+------------+---------------+

+--------+-------------+
| product|Total_Revenue|
+--------+-------------+
|  Laptop|       180000|
|  Tablet|       100000|
|Keyboard|        10000|
+--------+-------------+

+---------------+-----+
|delivery_status|count|
+---------------+-----+
|      Delivered|    5|
+---------------+-----+



In [14]:
spark_df.createOrReplaceTempView("orders")

spark.sql("""
SELECT product,
       SUM(order_amount) AS revenue
FROM orders
GROUP BY product
ORDER BY revenue DESC
""").show()

+--------+-------+
| product|revenue|
+--------+-------+
|  Laptop| 180000|
|  Tablet| 100000|
|Keyboard|  10000|
+--------+-------+

